[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonardijulia/AI4Good/blob/main/HandsOn_Part2/TerraTorch_HandsOn_Part2.ipynb)

# 🛠️ Setting up the environment
- Install terratorch (restart the environment as prompted)
- Import the necessary python libraries


In [ ]:
!pip install terratorch

In [ ]:
import os
import torch
import random
import numpy as np
from PIL import Image

from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import albumentations as A
from albumentations.pytorch import ToTensorV2

from lightning.pytorch import Trainer
from terratorch.models import EncoderDecoderFactory
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datamodules import GenericNonGeoSegmentationDataModule

# 📊 Import the dataset
MADOS? AI4SmallFarms?
Floods? BurnScars?

In [ ]:
# Load the dataset


### 🧹 Clean the dataset


### 🗺️ Visualize an example from the dataset

# 📑 Set up the datamodule

Datamodules are python classes that get your ready to feed to a DL model. TerraTorch offers four Generic Datamodules (Segmentation, Regression, Classification, and Multimodal) and a number of Specific Datamodules for concrete datasets (such as Sen1Floods11, HLS_BurnScars, Landslide4Sense, and more).

To set up a GFM pipeline we need to clearly define a Datamodule for our data (this entails we know the structure and characteristics of our data very well).

In [ ]:
# First we need the statistics of our training data to perform standardization
from terratorch.utils import compute_statistics # TerraTorch provides a built in function to calculate the statistics

# Define a temporary datamodule
temp_dm = GenericNonGeoSegmentationDataModule(
    batch_size=4,
    num_workers=2,
    num_classes=6,
    train_data_root="/content/demo/train/images",
    train_label_data_root="/content/demo/train/labels",
    val_data_root="/content/demo/test/images",
    val_label_data_root="/content/demo/test/labels",
    test_data_root="/content/demo/test/images",
    test_label_data_root="/content/demo/test/labels"
)
temp_dm.setup("fit")

# Compute the statistics
stats = compute_statistics(temp_dm.train_dataloader())
means, stds = stats['means'], stats['stds']

In [ ]:
# Define the datamodule with the transforms
datamodule = GenericNonGeoSegmentationDataModule(
    
)

datamodule.setup("fit")

In [ ]:
# We can use the built in .plot() function of the datamodule to visualize the data:

train_dataset = datamodule.train_dataset
train_dataset.plot(train_dataset[5])
plt.show()

# ⚙️ Set up the model

In [ ]:
from terratorch.registry import TERRATORCH_BACKBONE_REGISTRY

# We can check in the Backbone registry what pretrained models we can use;
# In this example we will use one of the Prithvi backbones
# For a detailed overview of Prithvi check out a previous AI4Good Workshop: https://www.youtube.com/watch?v=CB3FKtmuPI8

[backbone
 for backbone in TERRATORCH_BACKBONE_REGISTRY
 if ('prithvi') in backbone]

In [ ]:
# Set up the Backbone, Neck, Decoder parameters:
model_args = dict(
    # Backbone setup: pretrained Prithvi using the RGB bands
    backbone="",
    
    decoder="",

    # Neck setup:
    necks=[],
)

In [ ]:
# Define the TerraTorch task using the model arguments:
task = SemanticSegmentationTask(
    model_args,
    "EncoderDecoderFactory",
    loss="ce",
    lr=1e-4,
    ignore_index=-1,
    optimizer="AdamW",
    optimizer_hparams={"weight_decay": 0.05},
    freeze_backbone = False,
    freeze_decoder = False,
    plot_on_val = False,
)

In [ ]:
# Define the Lightning trainer:
trainer = Trainer(accelerator="cuda",
                  max_epochs=5,       # For demo purposes just 5 epochs
                  devices=1,
                  precision='16-mixed',
                  default_root_dir="/content/demo/" # Where the experiments' outputs will be
                )

# 🧠 Fine-tune the model

In [ ]:
# With everything set up, the training is run calling this one line of code:

trainer.fit(model=task, datamodule=datamodule)

# ✅ Test the fine-tuned model

In [ ]:
best_ckpt_path = "" # Load the checkpoint
trainer.test(model=task, datamodule=datamodule, ckpt_path=best_ckpt_path)

## 🖼️ Visualize the predictions

In [ ]:
model = SemanticSegmentationTask.load_from_checkpoint(
    best_ckpt_path,
    model_factory=task.hparams.model_factory,
    model_args=task.hparams.model_args,
).cuda() # Ensure the model is on the GPU

test_loader = datamodule.test_dataloader()
test_dataset = datamodule.test_dataset # Ensure test_dataset is available for plotting

with torch.no_grad():
    batch = next(iter(test_loader))
    images = batch["image"].to(model.device) # Move images to the same device as the model
    masks = batch["mask"].numpy()

    with torch.no_grad():
        outputs = model(images)

    preds = torch.argmax(outputs.output, dim=1).cpu().numpy()

for i in range(4):
    sample = {
        "image": batch["image"][i].cpu(),
        "mask": batch["mask"][i],
        "prediction": preds[i],
    }
    test_dataset.plot(sample)
    plt.show()

# 📜 Using the YAML file

In [ ]:
!pwd

In [ ]:
!wget https://raw.githubusercontent.com/leonardijulia/AI4Good/refs/heads/main/HandsOn_Part1/aerial_dubai.yaml # get the yaml file

In [ ]:
!terratorch fit -c /content/aerial_dubai.yaml

In [ ]:
!terratorch test -c /content/aerial_dubai.yaml --ckpt_path /content/demo/lightning_logs/version_3/checkpoints/epoch=4-step=70.ckpt